# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/J-Aamir/Capstone-Flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
if not os.path.exists("Capstone-Flyrank"):
    !git clone https://github.com/J-Aamir/Capstone-Flyrank.git
%cd Capstone-Flyrank

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df.trend_direction == "down").astype(int)
d = df[df.avg_position > 0].copy()

features = ['content_age_days','days_since_last_update','impressions_90d','avg_position',
            'ctr','word_count','search_volume']
clients = d['client_id'].unique()
train_clients, test_clients = train_test_split(clients, test_size=0.3, random_state=42)
train = d[d.client_id.isin(train_clients)]
test  = d[d.client_id.isin(test_clients)].copy()

Xtr, Xte = train[features].fillna(0), test[features].fillna(0)
ytr, yte = train['is_declining_label'].values, test['is_declining_label'].values
rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight='balanced',
                             random_state=42, n_jobs=-1).fit(Xtr, ytr)
test['model_score'] = rf.predict_proba(Xte)[:, 1]

benchmark = train.groupby('position_tier')['ctr'].mean()
test['benchmark_ctr'] = test['position_tier'].map(benchmark)
test['ctr_gap'] = (test['benchmark_ctr'] - test['ctr']).clip(lower=0)
print('Scored test set:', test.shape)


## 1. Ranked actions + reason codes

**Archetype -> action mapping**, combining the Week-5 decline-risk model and the Week-4 CTR-gap signal into one priority score (60% model risk, 40% normalized CTR gap):

| Reason code | Meaning | Action |
|---|---|---|
| `high_decline_risk_and_ctr_gap` | model flags high decline risk AND CTR is well below its position's benchmark | `refresh_content_and_review_snippet` |
| `high_decline_risk` | model flags high decline risk, CTR is fine | `refresh_content` |
| `ctr_below_position_benchmark` | decline risk is low, but CTR underperforms for its rank | `review_title_and_meta_description` |
| `low_priority` | neither signal is elevated | `monitor_only` |

**The decay/refresh insight** (consistent with the FlyRank research paper's Finding #4 and #8): older, stale pages that get refreshed recover meaningfully, while thin or never-updated old pages keep declining. This is why `refresh_content` targets high-risk pages specifically, rather than refreshing by age alone.

In [ ]:
def assign(row):
    if row.model_score >= 0.6 and row.ctr_gap > 0.3:
        return 'high_decline_risk_and_ctr_gap', 'refresh_content_and_review_snippet'
    elif row.model_score >= 0.6:
        return 'high_decline_risk', 'refresh_content'
    elif row.ctr_gap > 0.3:
        return 'ctr_below_position_benchmark', 'review_title_and_meta_description'
    else:
        return 'low_priority', 'monitor_only'

codes = test.apply(assign, axis=1, result_type='expand')
test['reason_code'] = codes[0]
test['action'] = codes[1]
test['priority_score'] = test['model_score'] * 0.6 + (test['ctr_gap'] / test['ctr_gap'].max()) * 0.4

queue = test.sort_values('priority_score', ascending=False)
print("Reason code distribution:")
print(queue['reason_code'].value_counts())
print()
print("Top 10 ranked actions:")
print(queue.head(10)[['content_id','priority_score','reason_code','action','trend_direction']].to_string(index=False))


## 2. Intended use and limits

**Who uses this:** a content/SEO editor doing weekly triage, with capacity to review roughly 20-50 pages per week. **What for:** deciding which pages to look at first, not an automatic publishing decision.

**Where it stops being valid:**
- Trained and validated on this specific 32-client (or client-holdout subset) snapshot; not validated on new clients, new industries, or data outside this time window.
- The decline label is a current-window proxy (`trend_direction`), not a guaranteed future outcome — a flagged page might recover on its own regardless of action.
- Precision@50 was 0.740 on honest client-grouped validation (Week 5/6) — meaning roughly 1 in 4 flagged pages in the top 50 will NOT actually be declining. This is decision-support, not a guarantee.
- Correlational, not causal: the model describes patterns associated with decline, it does not prove refreshing a page will fix it (same caution the FlyRank paper applies to its own findings).

In [ ]:
print(f"Validated Precision@50 (honest, client-grouped): 0.740")
print(f"Implied false-positive rate in top 50: {1 - 0.740:.1%}")
print("Intended user: content/SEO editor, weekly triage, capacity ~20-50 pages/week.")


## 3. Human review + the no-go list

**Before acting on any flagged page, a human must check:**
- Does the reason code's underlying data make sense for this specific page (not just the score)?
- Is there a known external cause (seasonal topic, a recent algorithm update, a competitor launch) the model wouldn't see?
- Would the suggested action (refresh vs. snippet edit) actually fit the page's real problem?

**No-go list — this should NEVER be automated:**
- Auto-publishing content changes without human edit/approval.
- Using the score to evaluate individual writers' or editors' performance.
- Treating a `low_priority` label as "this page is fine forever" — it means not urgent *right now*, not *permanently safe*.
- Using this tool's output as the sole justification for removing/deindexing content.
- Extrapolating these findings to clients or industries outside the training data without re-validation.

In [ ]:
no_go_list = [
    "Auto-publish content changes without human approval",
    "Use scores to evaluate individual writer/editor performance",
    "Treat 'low_priority' as permanently safe",
    "Use as sole justification for deindexing/removing content",
    "Apply to clients/industries outside the validated training data without re-validation",
]
for item in no_go_list:
    print("NO-GO:", item)


## 4. Monitoring / retrain triggers

**What would tell us the recommendations went stale:**
- Precision@50 on newly-labeled pages drops meaningfully below the validated 0.740 baseline over a rolling 4-week window.
- A new client/brand is onboarded (the model has never seen that content style or baseline traffic level — re-validate before trusting it there).
- The `position_tier` CTR benchmarks drift significantly (e.g. a Google SERP layout change shifts what "good CTR" looks like for a given rank).
- The underlying label distribution shifts sharply (e.g. the overall decline rate moves far from the ~54% observed in this data — could signal a platform-wide event, not individual page issues).

**Retrain cadence (suggested, not production-grade):** quarterly, or immediately if a monitoring trigger above fires.

In [ ]:
monitoring_triggers = {
    'precision_drop_threshold': 'Precision@50 falls below 0.65 on a rolling 4-week sample',
    'new_client_onboarded': 'Re-validate before trusting scores for any new client',
    'ctr_benchmark_drift': 'position_tier CTR benchmarks shift >20% from training values',
    'label_distribution_shift': 'Overall decline rate moves far from the ~54% baseline observed here',
}
for k, v in monitoring_triggers.items():
    print(f"{k}: {v}")


## 5. Exports for the paper

Writes the ranked queue (not committed, regenerated each run) and a reusable figure + metrics JSON (committed, since these are the "receipts" the paper's numbers trace back to).

In [ ]:
import json, os
import matplotlib.pyplot as plt

os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

# 1. The ranked queue CSV (NOT committed -- regenerated by the notebook, per the leak-guard)
out_cols = ['content_id','client_id','priority_score','reason_code','action','model_score',
            'ctr_gap','avg_position','impressions_90d','trend_direction']
queue[out_cols].to_csv('work/outputs/action_playbook_queue.csv', index=False)
print('Written: work/outputs/action_playbook_queue.csv')

# 2. A reusable figure (committed) -- reason code distribution
fig, ax = plt.subplots(figsize=(6,4))
queue['reason_code'].value_counts().plot(kind='barh', ax=ax)
ax.set_xlabel('Pages')
ax.set_title('Action playbook: reason code distribution')
plt.tight_layout()
plt.savefig('work/figures/w07_reason_code_distribution.png', dpi=150)
print('Written: work/figures/w07_reason_code_distribution.png')

# 3. Metrics JSON (committed) -- the receipts
metrics = {
    'validated_precision_at_50': 0.740,
    'validated_precision_at_20': 0.700,
    'baseline_precision_at_50': 0.580,
    'split_type': 'client_grouped',
    'n_test_clients': int(test['client_id'].nunique()),
    'n_test_rows': int(len(test)),
    'reason_code_counts': queue['reason_code'].value_counts().to_dict(),
}
with open('work/outputs/w07_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Written: work/outputs/w07_metrics.json')
print()
print(json.dumps(metrics, indent=2))


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.